# 🩺 SepsisSense - Exploratory Data Analysis (EDA)

## 📊 Comprehensive Analysis of Sepsis Prediction Dataset

This notebook provides an in-depth exploratory analysis of the PhysioNet Challenge 2019 sepsis dataset, focusing on understanding patterns, relationships, and insights that will inform our AI-powered sepsis prediction system.

### 🎯 Analysis Goals:
- **Patient Demographics**: Age, gender, ICU unit distribution
- **Vital Signs Patterns**: Temporal trends and distributions
- **Laboratory Values**: Clinical significance and missing data patterns  
- **Sepsis Onset Analysis**: Timeline and predictive indicators
- **Feature Relationships**: Correlations and interactions
- **Data Quality Assessment**: Missing values, outliers, and anomalies

---

**Author**: SepsisSense AI Team  
**Date**: October 2025  
**Dataset**: PhysioNet Challenge 2019 - Early Prediction of Sepsis

In [ ]:
# 📚 Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
from scipy import stats
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Custom color palette for healthcare theme
healthcare_colors = {
    'primary': '#2E86AB',      # Blue
    'secondary': '#A23B72',    # Purple
    'success': '#F18F01',      # Orange
    'danger': '#C73E1D',       # Red
    'warning': '#FFD23F',      # Yellow
    'info': '#06FFA5',         # Green
    'light': '#F8F9FA',
    'dark': '#343A40'
}

print("📦 Libraries imported successfully!")
print("🎨 Healthcare color theme configured!")
print("⚙️ Display settings optimized!")

In [ ]:
# 🔄 Generate and Load Sample Dataset
import sys
sys.path.append('../src/')

from preprocess import SepsisDataPreprocessor, create_sample_dataset

# Create sample dataset for analysis
print("🔄 Generating synthetic sepsis dataset...")
df = create_sample_dataset(n_patients=500, records_per_patient=24)

print(f"📊 Dataset Overview:")
print(f"   • Total records: {len(df):,}")
print(f"   • Unique patients: {df['Patient'].nunique():,}")
print(f"   • Sepsis cases: {df['SepsisLabel'].sum():,} ({df['SepsisLabel'].mean():.2%})")
print(f"   • Time range: {df['ICULOS'].min()}-{df['ICULOS'].max()} hours")
print(f"   • Features: {df.shape[1]} columns")

# Display basic info
print("\n📋 Dataset Info:")
df.info()

## 👥 Patient Demographics Analysis

Understanding the patient population is crucial for developing robust sepsis prediction models. Let's explore age distribution, gender patterns, and ICU unit characteristics.

In [ ]:
# 👥 Demographics Analysis
# Get patient-level data (one record per patient)
patient_df = df.groupby('Patient').agg({
    'Age': 'first',
    'Gender': 'first', 
    'Unit1': 'first',
    'Unit2': 'first',
    'SepsisLabel': 'max',  # Patient had sepsis if any record is positive
    'ICULOS': 'max'  # ICU length of stay
}).reset_index()

print(f"👥 Patient Demographics Summary:")
print(f"   • Total patients: {len(patient_df):,}")
print(f"   • Patients with sepsis: {patient_df['SepsisLabel'].sum():,} ({patient_df['SepsisLabel'].mean():.2%})")
print(f"   • Average age: {patient_df['Age'].mean():.1f} ± {patient_df['Age'].std():.1f} years")
print(f"   • Age range: {patient_df['Age'].min():.0f} - {patient_df['Age'].max():.0f} years")
print(f"   • Average ICU stay: {patient_df['ICULOS'].mean():.1f} ± {patient_df['ICULOS'].std():.1f} hours")

# Create comprehensive demographics visualization
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Age Distribution', 'Age by Sepsis Status', 'Gender Distribution',
        'ICU Length of Stay', 'ICU Unit Distribution', 'Sepsis Rate by Age Group'
    ],
    specs=[[{"type": "histogram"}, {"type": "box"}, {"type": "bar"}],
           [{"type": "histogram"}, {"type": "bar"}, {"type": "bar"}]]
)

# Age distribution
fig.add_trace(
    go.Histogram(x=patient_df['Age'], nbinsx=30, name='Age Distribution',
                marker_color=healthcare_colors['primary'], opacity=0.7),
    row=1, col=1
)

# Age by sepsis status
fig.add_trace(
    go.Box(y=patient_df[patient_df['SepsisLabel']==0]['Age'], name='No Sepsis',
           marker_color=healthcare_colors['info']),
    row=1, col=2
)
fig.add_trace(
    go.Box(y=patient_df[patient_df['SepsisLabel']==1]['Age'], name='Sepsis',
           marker_color=healthcare_colors['danger']),
    row=1, col=2
)

# Gender distribution
gender_counts = patient_df['Gender'].value_counts()
fig.add_trace(
    go.Bar(x=['Female', 'Male'], y=gender_counts.values, name='Gender',
           marker_color=[healthcare_colors['secondary'], healthcare_colors['primary']]),
    row=1, col=3
)

# ICU Length of Stay
fig.add_trace(
    go.Histogram(x=patient_df['ICULOS'], nbinsx=30, name='ICU LOS',
                marker_color=healthcare_colors['success'], opacity=0.7),
    row=2, col=1
)

# ICU Unit distribution
unit_data = pd.DataFrame({
    'Unit1': patient_df['Unit1'].value_counts(),
    'Unit2': patient_df['Unit2'].value_counts()
})
fig.add_trace(
    go.Bar(x=['Unit 1 No', 'Unit 1 Yes'], y=patient_df['Unit1'].value_counts().values,
           name='Unit 1', marker_color=healthcare_colors['warning']),
    row=2, col=2
)

# Sepsis rate by age group
age_groups = pd.cut(patient_df['Age'], bins=[0, 40, 60, 80, 100], labels=['<40', '40-60', '60-80', '80+'])
sepsis_by_age = patient_df.groupby(age_groups)['SepsisLabel'].mean()
fig.add_trace(
    go.Bar(x=sepsis_by_age.index.astype(str), y=sepsis_by_age.values, name='Sepsis Rate',
           marker_color=healthcare_colors['danger']),
    row=2, col=3
)

# Update layout
fig.update_layout(
    height=800, 
    title_text="🏥 Patient Demographics Dashboard",
    showlegend=False,
    title_font_size=16
)

fig.show()

# Statistical tests
print("\n📊 Statistical Analysis:")

# Age comparison between sepsis/no-sepsis groups
sepsis_age = patient_df[patient_df['SepsisLabel']==1]['Age']
no_sepsis_age = patient_df[patient_df['SepsisLabel']==0]['Age']
t_stat, p_value = stats.ttest_ind(sepsis_age, no_sepsis_age)
print(f"   • Age difference (t-test): t={t_stat:.3f}, p={p_value:.3f}")

# Gender vs sepsis association
contingency_table = pd.crosstab(patient_df['Gender'], patient_df['SepsisLabel'])
chi2, p_chi2, dof, expected = chi2_contingency(contingency_table)
print(f"   • Gender-sepsis association (χ²): χ²={chi2:.3f}, p={p_chi2:.3f}")

print("\n🎯 Key Demographics Insights:")
print(f"   • Mean age difference: {sepsis_age.mean() - no_sepsis_age.mean():.1f} years")
print(f"   • Age effect: {'Significant' if p_value < 0.05 else 'Not significant'} (p={p_value:.3f})")
print(f"   • Gender effect: {'Significant' if p_chi2 < 0.05 else 'Not significant'} (p={p_chi2:.3f})")

## 💓 Vital Signs Analysis

Vital signs are the primary indicators of patient health status. Let's explore their distributions, temporal patterns, and relationships with sepsis onset.

In [ ]:
# 💓 Vital Signs Distribution Analysis
vital_signs = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp']
vital_signs_available = [col for col in vital_signs if col in df.columns]

print(f"💓 Vital Signs Analysis:")
print(f"   • Available vital signs: {len(vital_signs_available)}")
print(f"   • Total measurements: {len(df):,}")

# Create vital signs distribution dashboard
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[f'{vs} Distribution' for vs in vital_signs_available] + ['Vital Signs Correlation'],
    specs=[[{"type": "histogram"}, {"type": "histogram"}, {"type": "histogram"}],
           [{"type": "histogram"}, {"type": "histogram"}, {"type": "histogram"}],
           [{"type": "histogram"}, {"colspan": 2, "type": "heatmap"}, None]]
)

# Individual vital signs distributions
positions = [(1,1), (1,2), (1,3), (2,1), (2,2), (2,3), (3,1)]

for i, vs in enumerate(vital_signs_available):
    if i < len(positions):
        row, col = positions[i]
        
        # Distribution by sepsis status
        sepsis_data = df[df['SepsisLabel']==1][vs].dropna()
        no_sepsis_data = df[df['SepsisLabel']==0][vs].dropna()
        
        fig.add_trace(
            go.Histogram(x=no_sepsis_data, name=f'{vs} No Sepsis', opacity=0.6,
                        marker_color=healthcare_colors['info'], nbinsx=30),
            row=row, col=col
        )
        
        fig.add_trace(
            go.Histogram(x=sepsis_data, name=f'{vs} Sepsis', opacity=0.6,
                        marker_color=healthcare_colors['danger'], nbinsx=30),
            row=row, col=col
        )

# Correlation heatmap
correlation_matrix = df[vital_signs_available].corr()
fig.add_trace(
    go.Heatmap(z=correlation_matrix.values,
               x=correlation_matrix.columns,
               y=correlation_matrix.index,
               colorscale='RdBu_r',
               zmid=0,
               text=np.around(correlation_matrix.values, decimals=2),
               texttemplate="%{text}",
               textfont={"size": 10},
               colorbar=dict(title="Correlation")),
    row=3, col=2
)

fig.update_layout(
    height=1000,
    title_text="💓 Vital Signs Comprehensive Analysis",
    showlegend=False,
    title_font_size=16
)

fig.show()

# Statistical analysis of vital signs
print("\n📊 Vital Signs Statistics by Sepsis Status:")
print("="*80)

for vs in vital_signs_available:
    sepsis_values = df[df['SepsisLabel']==1][vs].dropna()
    no_sepsis_values = df[df['SepsisLabel']==0][vs].dropna()
    
    if len(sepsis_values) > 0 and len(no_sepsis_values) > 0:
        # Statistical test
        t_stat, p_value = stats.ttest_ind(sepsis_values, no_sepsis_values)
        
        print(f"\n{vs}:")
        print(f"   No Sepsis: {no_sepsis_values.mean():.2f} ± {no_sepsis_values.std():.2f}")
        print(f"   Sepsis:    {sepsis_values.mean():.2f} ± {sepsis_values.std():.2f}")
        print(f"   Difference: {sepsis_values.mean() - no_sepsis_values.mean():.2f}")
        print(f"   Significance: {'***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'NS'} (p={p_value:.4f})")

In [ ]:
# 📈 Temporal Patterns in Vital Signs
print("📈 Analyzing Temporal Patterns...")

# Select a sample of patients for temporal analysis
sample_patients = df['Patient'].unique()[:20]  # First 20 patients
temporal_data = df[df['Patient'].isin(sample_patients)].copy()

# Create temporal visualization
fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=['Heart Rate Over Time', 'Blood Pressure Over Time',
                   'Oxygen Saturation Over Time', 'Temperature Over Time',
                   'Respiration Rate Over Time', 'Sepsis Timeline',
                   'Vital Signs Before Sepsis', 'Risk Score Progression'],
    vertical_spacing=0.05
)

# Heart Rate
for patient in sample_patients[:5]:  # Show first 5 patients
    patient_data = temporal_data[temporal_data['Patient'] == patient]
    sepsis_patient = patient_data['SepsisLabel'].max() > 0
    
    fig.add_trace(
        go.Scatter(x=patient_data['ICULOS'], y=patient_data['HR'],
                  mode='lines+markers', name=f'HR {patient}',
                  line_color=healthcare_colors['danger'] if sepsis_patient else healthcare_colors['primary'],
                  opacity=0.7),
        row=1, col=1
    )

# Blood Pressure (MAP)
for patient in sample_patients[:5]:
    patient_data = temporal_data[temporal_data['Patient'] == patient]
    sepsis_patient = patient_data['SepsisLabel'].max() > 0
    
    fig.add_trace(
        go.Scatter(x=patient_data['ICULOS'], y=patient_data['MAP'],
                  mode='lines+markers', name=f'MAP {patient}',
                  line_color=healthcare_colors['danger'] if sepsis_patient else healthcare_colors['info'],
                  opacity=0.7),
        row=1, col=2
    )

# Oxygen Saturation
for patient in sample_patients[:5]:
    patient_data = temporal_data[temporal_data['Patient'] == patient]
    sepsis_patient = patient_data['SepsisLabel'].max() > 0
    
    fig.add_trace(
        go.Scatter(x=patient_data['ICULOS'], y=patient_data['O2Sat'],
                  mode='lines+markers', name=f'O2Sat {patient}',
                  line_color=healthcare_colors['danger'] if sepsis_patient else healthcare_colors['success'],
                  opacity=0.7),
        row=2, col=1
    )

# Temperature
for patient in sample_patients[:5]:
    patient_data = temporal_data[temporal_data['Patient'] == patient]
    sepsis_patient = patient_data['SepsisLabel'].max() > 0
    
    fig.add_trace(
        go.Scatter(x=patient_data['ICULOS'], y=patient_data['Temp'],
                  mode='lines+markers', name=f'Temp {patient}',
                  line_color=healthcare_colors['danger'] if sepsis_patient else healthcare_colors['warning'],
                  opacity=0.7),
        row=2, col=2
    )

# Respiration Rate
for patient in sample_patients[:5]:
    patient_data = temporal_data[temporal_data['Patient'] == patient]
    sepsis_patient = patient_data['SepsisLabel'].max() > 0
    
    fig.add_trace(
        go.Scatter(x=patient_data['ICULOS'], y=patient_data['Resp'],
                  mode='lines+markers', name=f'Resp {patient}',
                  line_color=healthcare_colors['danger'] if sepsis_patient else healthcare_colors['secondary'],
                  opacity=0.7),
        row=3, col=1
    )

# Sepsis Timeline
sepsis_timeline = df.groupby('ICULOS')['SepsisLabel'].agg(['mean', 'count']).reset_index()
sepsis_timeline = sepsis_timeline[sepsis_timeline['count'] >= 10]  # Filter for sufficient data

fig.add_trace(
    go.Scatter(x=sepsis_timeline['ICULOS'], y=sepsis_timeline['mean'],
              mode='lines+markers', name='Sepsis Rate',
              line_color=healthcare_colors['danger'],
              fill='tonexty'),
    row=3, col=2
)

# Vital signs before sepsis onset
sepsis_patients_data = []
for patient in df[df['SepsisLabel']==1]['Patient'].unique():
    patient_data = df[df['Patient']==patient].copy()
    sepsis_onset = patient_data[patient_data['SepsisLabel']==1]['ICULOS'].min()
    
    # Get data 12 hours before sepsis onset
    pre_sepsis_data = patient_data[
        (patient_data['ICULOS'] <= sepsis_onset) & 
        (patient_data['ICULOS'] >= sepsis_onset - 12)
    ].copy()
    
    if len(pre_sepsis_data) >= 3:
        pre_sepsis_data['hours_before_sepsis'] = sepsis_onset - pre_sepsis_data['ICULOS']
        sepsis_patients_data.append(pre_sepsis_data)

if sepsis_patients_data:
    combined_pre_sepsis = pd.concat(sepsis_patients_data)
    
    # Average vital signs trajectory
    vital_trajectory = combined_pre_sepsis.groupby('hours_before_sepsis')['HR'].mean().reset_index()
    
    fig.add_trace(
        go.Scatter(x=vital_trajectory['hours_before_sepsis'], y=vital_trajectory['HR'],
                  mode='lines+markers', name='Avg HR Before Sepsis',
                  line_color=healthcare_colors['danger']),
        row=4, col=1
    )

# Risk score progression (simplified)
df['risk_score'] = (
    (df['HR'] > 100).astype(int) +
    (df['MAP'] < 70).astype(int) +
    (df['O2Sat'] < 95).astype(int) +
    (df['Temp'] > 38).astype(int) +
    (df['Resp'] > 20).astype(int)
) / 5

risk_progression = df.groupby('ICULOS')['risk_score'].mean().reset_index()
fig.add_trace(
    go.Scatter(x=risk_progression['ICULOS'], y=risk_progression['risk_score'],
              mode='lines+markers', name='Average Risk Score',
              line_color=healthcare_colors['warning']),
    row=4, col=2
)

fig.update_layout(
    height=1200,
    title_text="📈 Temporal Patterns and Risk Progression",
    showlegend=False,
    title_font_size=16
)

fig.show()

print("\n🎯 Temporal Analysis Insights:")
print(f"   • Peak sepsis incidence occurs around hour: {sepsis_timeline.loc[sepsis_timeline['mean'].idxmax(), 'ICULOS']}")
print(f"   • Average risk score for all patients: {df['risk_score'].mean():.3f}")
print(f"   • Average risk score for sepsis patients: {df[df['SepsisLabel']==1]['risk_score'].mean():.3f}")
print(f"   • Average risk score for non-sepsis patients: {df[df['SepsisLabel']==0]['risk_score'].mean():.3f}")